# Initialize models

Launch with the `ddpm` kernel. Builds:
1. the **evaluation model** — a FaceNet embedding extractor (`classify=False`, 512-d) saved to
   `eval_model/FaceNet_pytorch.tar`, used by `gather_results.py` to score ASR-L2; and
2. the per-scheme untrained **target classifier** `T_path` checkpoints that `train_reg.py` loads
   as a starting point: the facenet (vggface2) backbone + a fresh, **untrained** classifier head
   sized to each scheme's (protected) embedding.


In [ ]:
import os, sys
import torch
import torch.nn as nn

sys.path.insert(0, os.path.abspath('.'))  # BiDO/ holds model.py
import model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_CLASSES = 1000
OUT_DIR = 'target_model'
os.makedirs(OUT_DIR, exist_ok=True)
print('device:', device)

## Save eval model


In [ ]:
# Embedding evaluator E (FaceNet, classify=False -> 512-d embeddings). gather_results.py loads
# this from eval_model/FaceNet_pytorch.tar to score the embedding ASR.
eval_path = 'eval_model'
os.makedirs(eval_path, exist_ok=True)

E = model.InceptionResnetV1Unprotected(pretrained='vggface2', classify=False)
E.eval()
E = nn.DataParallel(E).to(device)
save_path = os.path.join(eval_path, 'FaceNet_pytorch.tar')
torch.save({'state_dict': E.state_dict()}, save_path)
print('saved eval model ->', save_path)


## Save (untrained) target models


In [ ]:
# scheme -> untrained checkpoint filename (matches train_reg.py's T_path:
#   FaceNet_pytorch_classify_<scheme>_untrained.tar, unprotected = FaceNet_pytorch_classify_untrained.tar)
SCHEMES = {
    'unprotected':               'FaceNet_pytorch_classify_untrained.tar',
    'facialfe_e8':               'FaceNet_pytorch_classify_facialfe_e8_untrained.tar',
    'facialfe_leech':            'FaceNet_pytorch_classify_facialfe_leech_untrained.tar',
    'facialfe_leech_randomised': 'FaceNet_pytorch_classify_facialfe_leech_randomised_untrained.tar',
    'ourfe':                     'FaceNet_pytorch_classify_ourfe_untrained.tar',
}

for scheme, fname in SCHEMES.items():
    if scheme == 'unprotected':
        net = model.InceptionResnetV1Unprotected(
            pretrained='vggface2', classify=True, num_classes=N_CLASSES, device=device)
        head_dim = net.logits.weight.shape  # (1000, 512)
    else:
        net = model.InceptionResnetV1PipeFE(
            scheme=scheme, pretrained='vggface2', classify=True, num_classes=N_CLASSES, device=device)
        head_dim = net.logits.weight.shape  # (1000, fe_out_dim), e.g. (1000, 504)
    # Wrap in DataParallel before saving so the state_dict keys ('module.*') match the
    # DataParallel-wrapped net that train_reg.py loads into (strict=False).
    net = nn.DataParallel(net).to(device)
    path = os.path.join(OUT_DIR, fname)
    torch.save({'state_dict': net.state_dict()}, path)
    print(f'{scheme:28s} logits head = {tuple(head_dim)}  ->  {path}')
    del net
    if device == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# Sanity check: reload one checkpoint into a fresh model (the way train_reg.py does).
scheme = 'facialfe_leech'
net = model.InceptionResnetV1PipeFE(scheme=scheme, pretrained=None, classify=True,
                                    num_classes=N_CLASSES, device=device)
net = nn.DataParallel(net).to(device)
ckp = torch.load(os.path.join(OUT_DIR, SCHEMES[scheme]), map_location=device)
missing, unexpected = net.load_state_dict(ckp['state_dict'], strict=False)
print('missing keys:', len(missing), '| unexpected keys:', len(unexpected))
print('reload OK; head =', tuple(net.module.logits.weight.shape))